# Inspect SWE Target

QA/QC of the combined snow-water-equivalent calibration target
written by `targets/swe.py` to `<project>/targets/swe_targets.nc` (and
the optional `swe_targets_nn_filled.nc` companion).

The SWE target is a per-HRU per-**day** `(lower_bound, upper_bound)`
range in **inches** (PRMS `pkwater_equiv` units), formed as the
NaN-aware min / max across up to four sources on a shared fabric:

- Daymet V4 R1 `swe` (kg/m² ≡ mm, daily) → inches.
- SNODAS `swe` (kg/m² ≡ mm, daily) → inches.
- ERA5-Land `sd` (m, instantaneous daily snapshot) → mm → inches.
- Margulis WUS-SR `SWE` (m, daily) → mm → inches. Carries
  `fabric_scope.fabrics: [or]` in `catalog/sources.yml` — present
  only on the OR fabric, dropped automatically on every other fabric
  by `targets/swe.py:_filter_sources_by_fabric_scope`. On the gfv2
  fabric the SWE bound is therefore a **3-source** combination
  (Daymet + SNODAS + ERA5-Land).

The accompanying `n_sources` flag (0 / 1 / 2 / 3 / 4) records how many
sources contributed at each cell — the bound is NaN only when
`n_sources == 0`. The `nn_filled` companion file reuses the bounds
but additionally fills NaN HRUs from up to 10 nearest neighbours; the
`nn_filled` int8 flag (0 / 1) marks which HRU-times were filled.

This notebook checks:

- File schema and global metadata (period, fabric SHA, source string,
  `n_sources_count`).
- Per-time `n_sources` coverage and where the gaps fall geographically.
- `lower_bound`, `upper_bound`, and range size at peak winter day
  (`TARGET_DATE`, default 2010-03-01).
- CONUS area-weighted mean lower / upper time series (monthly-resampled
  for legibility — the underlying NC is daily).
- Representative-HRU daily series across four snow-bearing regions.
- NN-fill: which HRU-times were filled and whether the filled values
  preserve the lower / upper distribution.
- Order-of-magnitude sanity check against published peak SWE at
  representative snow-belt locations.

Companion to:

- `inspect_aggregated_snodas.ipynb`, `inspect_aggregated_daymet.ipynb`,
  `inspect_aggregated_era5_land_sd.ipynb`, and (after #101's Margulis
  aggregate sub-task) `inspect_aggregated_margulis_wus_sr.ipynb` —
  per-source HRU aggregates before multi-source combination.

## Conventions

- HRU dim name follows `fabric.id_col` from the project config
  (e.g. `nat_hru_id` for the GFv2 fabric, `nhm_id` elsewhere).
- Units are read from the **target NC variable attrs** (`inches`).
- `TARGET_DATE` (set below) drives the at-time choropleth panels.
  Default is **March 1**, near peak CONUS SWE — the cleanest spread
  across sources because every Sierra / Rockies / Cascades / Adirondack
  HRU has finite snowpack. Switch to a summer date to inspect the
  glacier-only / NaN-everywhere regime instead.
- The SWE NC is **daily**. The full time series is ~365 timesteps
  per year — for CONUS-mean plots we resample to **monthly mean** for
  legibility; representative-HRU plots stay daily so peak snowpack and
  melt-out events stay visible.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    area_weighted_series,
    discover_target_nc,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    n_sources_per_time,
    nan_hru_count,
    open_target_nc,
    plot_categorical_choropleth,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/targets/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

TARGET = "swe"
TARGET_DATE = "2010-03-01"  # near-peak CONUS SWE

# Four snow-bearing regions spanning the major CONUS snowpack regimes.
REPRESENTATIVE_POINTS = {
    "Sierra Nevada (deep maritime snowpack)": (-119.0, 38.0),
    "Colorado Rockies (alpine continental)": (-106.8, 39.5),
    "Cascades (PNW maritime, heavy SWE)": (-121.5, 47.0),
    "Adirondacks (eastern shallow snowpack)": (-74.0, 44.0),
}

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)
id_dim = fabric_cfg["id_col"]

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
fabric_path = fabric_cfg['path']
print(f"Fabric: {fabric_path} ({len(fabric)} HRUs, id_col={id_dim!r})")
print(f"Target date: {TARGET_DATE}")

## Open and summarise the target NCs

`discover_target_nc` finds both the unfilled (`swe_targets.nc`) and
NN-filled (`swe_targets_nn_filled.nc`) variants if they exist. Either
may be `None` — this cell skips with a clear message and downstream
cells iterate only over what was loaded.

In [ ]:
raw_path, filled_path = discover_target_nc(project_dir, TARGET)

if raw_path is None:
    print(f"SKIP: {TARGET}_targets.nc not found at {project_dir / 'targets'}.")
    print("      Run `pixi run run-swe -- --project-dir <project>` first.")
    raise SystemExit

ds_raw = open_target_nc(raw_path)
print(f"Loaded raw target:    {raw_path.name}  ({raw_path.stat().st_size / 1e6:.1f} MB)")

ds_filled = None
if filled_path is not None:
    ds_filled = open_target_nc(filled_path)
    print(
        f"Loaded NN-filled:     {filled_path.name}  "
        # f"({filled_path.stat().st_size / 1e6:.1f} MB)"
    )
else:
    print("NN-filled variant absent (set `nn_fill: true` in config to produce it).")

## Schema and global metadata

The global attrs carry the provenance you'd want at calibration time:
the period, the fabric SHA-256 (so a downstream consumer can detect a
fabric swap), the comma-separated `source` string, and
`n_sources_count` (the number of sources the builder actually used —
3 on non-Oregon fabrics, 4 on the OR fabric where Margulis is in scope).
Per-variable attrs carry the units (`inches` for the bounds, `1` for
the diagnostic flags).

In [ ]:
print(ds_raw)
print()
print("=== Global attrs ===")
for k, v in ds_raw.attrs.items():
    print(f"  {k:<20} {v}")

print()
print("=== Per-variable units ===")
for v in ("lower_bound", "upper_bound", "n_sources"):
    units = ds_raw[v].attrs.get("units", "(no units attr)")
    long_name = ds_raw[v].attrs.get("long_name", "")
    print(f"  {v:<14} units={units!r}  long_name={long_name!r}")

## Per-time coverage

`n_sources_per_time` returns the per-day count of HRUs at each flag
value (0 / 1 / 2 / 3 / 4). Daily resolution over a multi-year run is
noisy, so we **resample to monthly mean** before plotting.

Two diagnostics:

- The `n=0` column is the count of all-NaN HRUs at that timestep —
  these are the cells the NN-fill targets.
- A jump in `n=2` (or drop in `n=3` / `n=4`) at a particular date
  flags a source whose period ended there. Daymet runs through 2024,
  SNODAS through present, ERA5-Land through near-present. Margulis
  WUS-SR runs through 2017 (and only on the OR fabric).

In [ ]:
cov = n_sources_per_time(ds_raw)
# Daily series is too dense for a multi-year plot; resample to month-start mean.
cov_monthly = cov.resample("MS").mean()
print(cov_monthly.describe().T[["mean", "std", "min", "max"]])

fig, ax = plt.subplots(figsize=(11, 4))
cov_monthly.plot(ax=ax)
ax.set_xlabel("Time (monthly mean of daily counts)")
ax.set_ylabel("HRU count")
ax.set_title(f"Per-day n_sources distribution — {TARGET} target (monthly mean)")
ax.legend(title="flag value", loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_coverage_timeseries")
plt.show()

## n_sources map at TARGET_DATE

Categorical choropleth — colours match the `flag_values` attr on the
`n_sources` variable. NaN HRUs (where the time slice is missing) plot
in light grey; the legend reports the count of HRUs at each flag value
so coverage anomalies are visible at a glance.

In [ ]:
ns = ds_raw["n_sources"].sel(time=TARGET_DATE).to_pandas()

categories = {
    0: ("0 sources (all NaN)", "crimson"),
    1: ("1 source", "khaki"),
    2: ("2 sources", "lightgreen"),
    3: ("3 sources", "darkgreen"),
    4: ("4 sources", "midnightblue"),
}

fig, ax = plt.subplots(figsize=(11, 7))
plot_categorical_choropleth(
    ax,
    fabric,
    ns,
    categories=categories,
    title=f"n_sources at {TARGET_DATE} — {TARGET} target",
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_n_sources_map")
plt.show()

## Lower / upper / range maps at TARGET_DATE

Three side-by-side panels:

- `lower_bound` — NaN-aware min across the contributing sources (inches).
- `upper_bound` — NaN-aware max across the contributing sources (inches).
- `range = upper - lower` — the per-HRU per-day spread the NHM
  calibrator is asked to land inside. Wide ranges = poor cross-source
  agreement (common in deep maritime snowpack where MM/SAT/reanalysis
  diverge); tight ranges = consensus.

Colour scale is anchored on the 2nd / 98th percentile of `upper_bound`
so the long tail of deep-snow HRUs doesn't flatten the rest of the map.

In [ ]:
lb = ds_raw["lower_bound"].sel(time=TARGET_DATE).to_pandas()
ub = ds_raw["upper_bound"].sel(time=TARGET_DATE).to_pandas()
rng = ub - lb

ub_finite = ub.dropna().values
vmin = float(np.percentile(ub_finite, 2))
vmax = float(np.percentile(ub_finite, 98))

units = ds_raw["lower_bound"].attrs.get("units", "inches")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
plot_hru_choropleth(
    axes[0], fabric, lb, vmin=vmin, vmax=vmax, cmap="Blues",
    title=f"lower_bound\n{TARGET_DATE} | {units}", units=units,
)
plot_hru_choropleth(
    axes[1], fabric, ub, vmin=vmin, vmax=vmax, cmap="Blues",
    title=f"upper_bound\n{TARGET_DATE} | {units}", units=units,
)
plot_hru_choropleth(
    axes[2], fabric, rng, vmin=0, vmax=vmax, cmap="OrRd",
    title=f"range = upper - lower\n{TARGET_DATE} | {units}", units=units,
)
fig.suptitle(f"{TARGET} target bounds — {TARGET_DATE}", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_bounds_map")
plt.show()

print(f"lower CONUS area-weighted mean: {area_weighted_mean(lb, fabric):,.2f} {units}")
print(f"upper CONUS area-weighted mean: {area_weighted_mean(ub, fabric):,.2f} {units}")
print(f"range CONUS area-weighted mean: {area_weighted_mean(rng, fabric):,.2f} {units}")

## NaN HRU coverage

HRUs where `n_sources == 0` (and therefore `lower_bound` /
`upper_bound` are NaN). These are the cells that nearest-neighbour
fill targets. For SWE this is usually a small footprint at peak
winter (when every source has finite snowpack on the snow-bearing
fabric and zero elsewhere), but expands in summer when SNODAS
explicitly drops zero pixels and only Daymet/ERA5-Land/Margulis carry
the no-snow signal.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
plot_nan_hrus(
    ax, fabric, lb,
    title=(
        f"NaN HRUs (red) — {TARGET_DATE} | "
        f"{nan_hru_count(lb)} of {len(fabric)} "
        f"({100 * nan_hru_count(lb) / len(fabric):.2f}%)"
    ),
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_nan_map")
plt.show()

## CONUS area-weighted lower / upper time series (monthly mean)

`area_weighted_series` reduces the (time, hru) bound arrays to a
per-timestep CONUS-mean, weighted by EPSG:5070 polygon area. Daily
resolution is too dense to plot legibly over multi-year periods, so
we resample the resulting series to **monthly mean** before plotting.

Lower / upper plotted together form an envelope — its width is the
average cross-source disagreement nationally. Watch for:

- A strong seasonal cycle (peak ~Mar, near-zero ~Aug) — SWE is the
  most strongly seasonal of all six targets.
- Envelope expansion during accumulation / melt — the sources
  disagree most about timing and magnitude of state changes.

In [ ]:
lb_series = area_weighted_series(ds_raw["lower_bound"], fabric, id_dim)
ub_series = area_weighted_series(ds_raw["upper_bound"], fabric, id_dim)
lb_m = lb_series.resample("MS").mean()
ub_m = ub_series.resample("MS").mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(
    lb_m.index, lb_m.values, ub_m.values,
    color="steelblue", alpha=0.25, label="lower–upper envelope",
)
ax.plot(lb_m.index, lb_m.values, color="steelblue", lw=1, label="lower (monthly mean)")
ax.plot(ub_m.index, ub_m.values, color="darkblue", lw=1, label="upper (monthly mean)")
ax.set_xlabel("Time")
ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
ax.set_title(f"{TARGET} target — CONUS area-weighted bounds (daily → monthly mean)")
ax.legend()
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_conus_series")
plt.show()

## Representative HRU time series (daily)

Four snow-bearing regions; `lookup_hrus_by_points` uses `gpd.sjoin` to
resolve each (lon, lat) to the containing HRU. Plotted at native
**daily** cadence so peak accumulation, mid-winter rain-on-snow
drawdowns, and melt-out timing stay visible. The lower / upper envelope
at each point is the per-day spread the calibrator targets locally.

In [ ]:
rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
print("Representative HRUs:", rep_hrus)

lb_at = ds_raw["lower_bound"].sel({id_dim: list(rep_hrus.values())})
ub_at = ds_raw["upper_bound"].sel({id_dim: list(rep_hrus.values())})

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
    lb_h = lb_at.sel({id_dim: hru_id}).values
    ub_h = ub_at.sel({id_dim: hru_id}).values
    t = pd.DatetimeIndex(lb_at.time.values)
    ax.fill_between(t, lb_h, ub_h, color="steelblue", alpha=0.25)
    ax.plot(t, lb_h, color="steelblue", lw=0.7, label="lower")
    ax.plot(t, ub_h, color="darkblue", lw=0.7, label="upper")
    ax.set_title(f"{label} (HRU {hru_id})")
    ax.set_ylabel(f"swe ({units})")
    ax.legend(fontsize=8)
fig.suptitle(f"{TARGET} target at representative HRUs (daily)", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_representative_series")
plt.show()

## NN-fill comparison

The companion `swe_targets_nn_filled.nc` reuses the same bounds but
fills NaN HRUs from up to `nn_max_candidates=10` nearest neighbours
(see `targets/_common.py:nn_fill_hru_nans_by_time`). The `nn_filled`
int8 flag marks each HRU-time as `0` (not filled) or `1` (filled).

Two checks:

- **Filled-flag map at TARGET_DATE** — should match the NaN-HRU map
  one-for-one (any NaN that *can* be filled gets a `1`).
- **Distribution preservation (monthly mean)** — the area-weighted
  CONUS mean of the filled bounds should track the unfilled mean
  closely. A large gap indicates an over-aggressive fill (donor HRUs
  too far from the gap).

In [ ]:
if ds_filled is None:
    print("SKIP: NN-filled variant not present.")
else:
    flag = ds_filled["nn_filled"].sel(time=TARGET_DATE).to_pandas()
    n_filled = int((flag == 1).sum())
    print(
        f"NN-filled at {TARGET_DATE}: {n_filled} HRUs filled "
        f"({100 * n_filled / len(fabric):.2f}%)"
    )

    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    plot_categorical_choropleth(
        axes[0], fabric, flag,
        categories={0: ("not filled", "lightgrey"), 1: ("filled", "crimson")},
        title=f"nn_filled flag at {TARGET_DATE}",
    )
    lb_f = ds_filled["lower_bound"].sel(time=TARGET_DATE).to_pandas()
    diff = (lb_f - lb).abs().fillna(lb_f)  # NaN in raw -> show filled value
    plot_hru_choropleth(
        axes[1], fabric, diff, vmin=0, vmax=vmax / 4, cmap="OrRd",
        title=f"|lower_bound_filled - lower_bound_raw| at {TARGET_DATE}",
        units=units,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_map")
    plt.show()

    lb_f_series = area_weighted_series(ds_filled["lower_bound"], fabric, id_dim).resample("MS").mean()
    ub_f_series = area_weighted_series(ds_filled["upper_bound"], fabric, id_dim).resample("MS").mean()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(lb_m.index, lb_m.values, color="steelblue", lw=1, label="lower (raw, monthly mean)")
    ax.plot(lb_f_series.index, lb_f_series.values, color="steelblue", lw=1, ls="--", label="lower (NN-filled, monthly mean)")
    ax.plot(ub_m.index, ub_m.values, color="darkblue", lw=1, label="upper (raw, monthly mean)")
    ax.plot(ub_f_series.index, ub_f_series.values, color="darkblue", lw=1, ls="--", label="upper (NN-filled, monthly mean)")
    ax.set_xlabel("Time")
    ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
    ax.set_title(f"{TARGET} target — raw vs NN-filled CONUS series (monthly mean)")
    ax.legend()
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_series")
    plt.show()

## Order-of-magnitude sanity check

Peak-winter SWE at representative snow-belt locations should match
well-known climatology:

- **Sierra Nevada (mid-elevation)** — typical March-1 SWE 20–60 in.
- **Colorado Rockies (alpine)** — typical March-1 SWE 10–25 in.
- **Cascades (windward)** — typical March-1 SWE 30–80 in.
- **Adirondacks** — typical March-1 SWE 4–12 in.

Per `feedback_validate_magnitudes`: an order-of-magnitude miss is a
smoking gun for a unit-conversion bug (mm vs in vs m, or accumulated
vs instantaneous). A factor-of-2 miss is plausible inter-source
disagreement and worth keeping in the lower–upper envelope.

In [ ]:
print(f"Sanity check at {TARGET_DATE} — lower / upper SWE at representative HRUs")
print(f"{'Region':<48} {'lower (in)':>12} {'upper (in)':>12}")
print("-" * 76)
for label, hru_id in rep_hrus.items():
    lb_h = float(ds_raw["lower_bound"].sel({id_dim: hru_id}).sel(time=TARGET_DATE).values)
    ub_h = float(ds_raw["upper_bound"].sel({id_dim: hru_id}).sel(time=TARGET_DATE).values)
    print(f"{label:<48} {lb_h:>12.2f} {ub_h:>12.2f}")

print()
print(f"CONUS area-weighted lower_bound at {TARGET_DATE}: {area_weighted_mean(lb, fabric):.4f} {units}")
print(f"CONUS area-weighted upper_bound at {TARGET_DATE}: {area_weighted_mean(ub, fabric):.4f} {units}")

In [ ]:
ds_raw.close()
if ds_filled is not None:
    ds_filled.close()